In [1]:

import pretty_midi
import numpy as np
import pandas as pd
from pathlib import Path


In [2]:
midi_paths = {
    "Pretrained": Path("../outputs/pretrained/unprompted.mid"),
    "Fine-tuned": Path("../outputs/finetuned/finetuned_piano.mid"),
}

def analyze_midi(path):
    midi = pretty_midi.PrettyMIDI(str(path))
    notes = [
        note
        for instrument in midi.instruments
        for note in instrument.notes
    ]

    if not notes:
        return {"Error": "No notes found"}

    pitches = np.array([n.pitch for n in notes])
    durations = np.array([n.end - n.start for n in notes])
    onsets = np.array(sorted(n.start for n in notes))

    duration_seconds = max(n.end for n in notes)

    return {
        "Notes": len(notes),
        "Duration (s)": round(duration_seconds, 2),
        "Note density (notes/s)": round(
            len(notes) / duration_seconds, 2
        ),
        "Pitch range": f"{pitches.min()}–{pitches.max()}",
        "Mean pitch": round(pitches.mean(), 2),
        "Mean note duration (s)": round(durations.mean(), 3),
        "Median note duration (s)": round(np.median(durations), 3),
        "Mean inter-onset interval (s)": round(
            np.diff(onsets).mean(), 3
        ) if len(onsets) > 1 else None,
    }


In [3]:

results = {}

for name, path in midi_paths.items():
    if not path.exists():
        print(f"Missing file: {path.resolve()}")
        continue

    results[name] = analyze_midi(path)

comparison = pd.DataFrame(results).T
display(comparison)


,Notes,Duration (s),Note density (notes/s),Pitch range,Mean pitch,Mean note duration (s),Median note duration (s),Mean inter-onset interval (s)
Pretrained,1064,487.06,2.18,32–104,56.4,0.379,0.312,0.456
Fine-tuned,1392,181.88,7.65,32–85,64.14,0.223,0.125,0.13


In [4]:
def analyze_bars(path):
    midi = pretty_midi.PrettyMIDI(str(path))

    # Estimate the number of bars using the MIDI's time signature
    time_signature = midi.time_signature_changes[0]
    beats_per_bar = time_signature.numerator

    # Use the first tempo to estimate bar duration
    tempo = midi.get_tempo_changes()[1][0]
    seconds_per_bar = beats_per_bar * 60 / tempo

    notes = [
        note
        for instrument in midi.instruments
        for note in instrument.notes
    ]

    duration = max(note.end for note in notes)
    num_bars = duration / seconds_per_bar

    return {
        "Estimated bars": round(num_bars, 2),
        "Notes per bar": round(len(notes) / num_bars, 2),
        "Seconds per bar": round(seconds_per_bar, 2),
        "Initial tempo (BPM)": round(tempo, 2),
        "Time signature": (
            f"{time_signature.numerator}/"
            f"{time_signature.denominator}"
        ),
    }

bar_results = {}

for name, path in midi_paths.items():
    if path.exists():
        bar_results[name] = analyze_bars(path)

bar_comparison = pd.DataFrame(bar_results).T
display(bar_comparison)

,Estimated bars,Notes per bar,Seconds per bar,Initial tempo (BPM),Time signature
Pretrained,243.53,4.37,2.0,120.0,4/4
Fine-tuned,90.94,15.31,2.0,120.0,4/4


In [5]:

from collections import Counter
import numpy as np
import pandas as pd
import pretty_midi

def analyze_diversity(path, pattern_length=4):
    midi = pretty_midi.PrettyMIDI(str(path))

    notes = [
        note
        for instrument in midi.instruments
        for note in instrument.notes
    ]

    notes.sort(key=lambda n: (n.start, n.pitch))
    pitches = [note.pitch for note in notes]
    pitch_classes = [pitch % 12 for pitch in pitches]

    patterns = [
        tuple(pitches[i:i + pattern_length])
        for i in range(len(pitches) - pattern_length + 1)
    ]

    pattern_counts = Counter(patterns)
    repeated_patterns = sum(
        count for count in pattern_counts.values() if count > 1
    )

    pc_counts = np.bincount(pitch_classes, minlength=12)
    pc_probs = pc_counts[pc_counts > 0] / len(pitch_classes)
    pitch_class_entropy = -np.sum(pc_probs * np.log2(pc_probs))

    return {
        "Unique pitches": len(set(pitches)),
        "Pitch classes used": len(set(pitch_classes)),
        "Pitch-class entropy (bits)": round(pitch_class_entropy, 3),
        "Unique 4-note patterns": len(pattern_counts),
        "Repeated 4-note patterns": sum(
            count > 1 for count in pattern_counts.values()
        ),
        "Notes in repeated patterns (%)": round(
            100 * repeated_patterns / len(pitches), 2
        ),
    }

diversity_results = {}

for name, path in midi_paths.items():
    if path.exists():
        diversity_results[name] = analyze_diversity(path)

diversity_comparison = pd.DataFrame(diversity_results).T
display(diversity_comparison)


,Unique pitches,Pitch classes used,Pitch-class entropy (bits),Unique 4-note patterns,Repeated 4-note patterns,Notes in repeated patterns (%)
Pretrained,57.0,12.0,2.882,961.0,70.0,15.98
Fine-tuned,38.0,12.0,3.016,1243.0,101.0,17.74
